# SIMT 核函数

## 概述

前面两节我们理解了 SIMT 的硬件架构和线程组织方式。本节进入实战：**如何把这套并行模型写成真正的代码？** 也就是如何编写 SIMT 算子的核心——核函数（Kernel）。

核函数是能在 AI 处理器（NPU）上并行执行、并由主机端（Host）发起调用的函数。它被设计为由大量线程同时并行运行，每个线程拥有独立的寄存器，协同完成海量数据的计算。本节会带你走完一个核函数的完整写法：从定义、调用，到用内置变量定位数据，并提醒一个实际开发中容易忽略的细节。

### 前置要求

- 已理解 Grid、Thread Block、Thread、`gridDim`、`blockDim`、`blockIdx` 和 `threadIdx` 的含义。
- 具备基础 C/C++ 函数定义和指针概念，能够阅读简单函数声明。
- 本小节为理论讲解，示例代码用于理解语法结构，不依赖在线硬件环境执行。

### 学习目标

学完本小节后，你应该能够：

- 写出核函数定义的基本形式，说明 `__global__` 和 `void` 返回类型两条规则。
- 说明 host 侧函数、核函数、device 侧函数之间的调用关系。
- 理解核函数调用符 `<<<...>>>` 中 4 个执行配置参数的含义。
- 说明执行配置参数与核函数内置变量之间的对应关系。
- 读写一个完整的一维 SIMT 核函数，并理解为什么需要边界检查。

### 小节内容

- 核函数是什么
- 核函数的定义
- 三类函数的调用关系
- 核函数的调用与执行配置
- 内置变量与执行配置的对应关系
- 一个完整的向量加法实例
- 一个容易忽略的细节：边界检查

### 核函数是什么

Ascend C 支持开发者自定义核函数来扩展 C++。和普通 C/C++ 函数最大的不同在于：普通函数在一个线程里顺序执行一次，而**核函数一旦启动，就会由成千上万个线程同时并行执行同一份代码**，共同完成数据计算任务。

结合前两节，可以从三个视角理解核函数：

- **硬件视角**：核函数运行在 AI 处理器上，线程会使用寄存器、Unified Buffer、Global Memory 等资源。
- **线程视角**：核函数启动后，线程按 Grid 和 Thread Block 的层次组织起来并发执行。
- **编程视角**：核函数是 host 侧通过特殊调用语法 `<<<...>>>` 启动的 device 侧并行入口。

接下来就从「怎么定义」开始。

### 核函数的定义

编译器提供特定的**函数类型限定符**来定义核函数，编译器能识别这个函数需要在 Device 侧编译，并允许它从 Host 侧被调用。基本形式如下：

```cpp
// 核函数定义示例
__global__ void vec_add(float* x, float* y, float* z)
{
    // 具体的计算逻辑
    uint32_t i = blockIdx.x * blockDim.x + threadIdx.x;
    z[i] = x[i] + y[i];
}
```

定义核函数要遵循以下规则：

1. 使用函数类型限定符 `__global__`，标识它是一个核函数。
2. 核函数的返回类型必须是 `void`（计算结果通过出参的方式写到内存中，而不是 return）。

这里要特别理解第 2 条「返回 void」：因为核函数会由成千上万个线程并行执行，没法用单一返回值表达所有线程的结果，所以需要让每个线程把自己算出的结果**写到指针指向的 Global Memory 或共享内存**。上面示例中的 `float* z` 就是这样的输出指针。

### 三类函数的调用关系

一个算子程序中的函数可以分为三类，它们的执行位置和调用关系不同：

| 函数类别 | 执行位置 | 类型限定符 | 调用关系 |
| --- | --- | --- | --- |
| host 侧执行函数 | Host 侧 | 无 | 可调用其它 host 函数，也可通过 `<<<...>>>` 调用核函数 |
| 核函数 | Device 侧 | `__global__` | 可调用除核函数外的其它 device 侧函数 |
| device 侧执行函数 | Device 侧 | `__aicore__` | 可调用同类的其它 device 侧函数 |

下图展示了三类函数之间的调用关系：

![三类函数的调用关系](images/kernel_function_call_relationship.svg)

几个要点：

- Host 侧函数是整个流程的起点，负责准备数据、配置参数，再用 `<<<...>>>` 发起核函数调用。
- 核函数是 Host 进入 Device 并行计算的**唯一入口**，但核函数**不能再调用另一个核函数**。
- 核函数内部可以调用用 `__aicore__` 标识的普通 device 侧执行函数，把计算逻辑拆分复用。

### 核函数的调用与执行配置

Host 侧通过核函数调用符 `<<<...>>>` 来调用核函数。基本形式如下：

```cpp
kernel_name<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>(argument_list);
```

可以把它拆成两部分理解：

- `<<<...>>>`：**执行配置**，告诉硬件用多少线程块、每块多少线程、动态共享内存多大、关联哪个 stream。
- `(argument_list)`：和普通函数一样的参数列表，传给核函数内部使用。

这正是核函数和普通 C/C++ 函数最直观的区别——普通函数调用只写 `func(args)`，核函数调用还必须显式给出**并行执行配置**。

`<<<...>>>` 里的 4 个参数含义如下：

| 参数 | 类型 | 含义 |
| --- | --- | --- |
| `blocks_per_grid` | int 或 dim3 | 网格的维度与规模，三维乘积 = 线程块总数，不得大于 65535。对应 `gridDim` |
| `threads_per_block` | int 或 dim3 | 每个线程块的维度与规模，三维乘积 = 每块线程数，不得大于 1024（默认，可配置）。对应 `blockDim` |
| `dyn_ubuf_size` | size_t | 为每个线程块动态分配的共享内存字节数，默认 0 |
| `stream` | aclrtStream | 关联的流，用于维护异步操作的执行顺序 |

初学阶段最该先掌握前两个：`blocks_per_grid` 决定有多少线程块，`threads_per_block` 决定每个线程块多少线程，二者共同决定核函数启动后的并行规模。动态分配共享内存的用法和配置最大线程数的方法将在下一章节详细介绍，此处不展开。

### 内置变量与执行配置的对应关系

核函数启动后，每个线程在函数内可以直接读取**内置变量**，从而知道自己是谁、该处理哪份数据。这些内置变量的取值，正是由前面 `<<<...>>>` 执行配置决定的，两者一一对应：

| 内置变量 | 含义 | 由哪个执行配置参数决定 |
| --- | --- | --- |
| `gridDim` | 网格在各维度上由多少个线程块构成 | `blocks_per_grid` |
| `blockDim` | 每个线程块的线程数 | `threads_per_block` |
| `blockIdx` | 当前线程块在网格中的索引 | 运行时由硬件分配 |
| `threadIdx` | 当前线程在所属线程块内的索引 | 运行时由硬件分配 |

简单说：`gridDim` 和 `blockDim` 是开发者在调用时**配置进去的规模**，而 `blockIdx` 和 `threadIdx` 是硬件在运行时**分配给每个线程的身份编号**。

至于如何用这几个变量组合出全局数据索引（`blockIdx.x * blockDim.x + threadIdx.x`），在上一章《线程架构》中已有详细讲解，这里不再展开。下面直接看它在核函数里的实际用法。

### 一个完整的向量加法实例

把前面所有知识点串起来，来看一个完整的核函数是怎么写的。以「向量相加 `z[i] = x[i] + y[i]`」为例，它包含**核函数实现**和 **Host 侧调用**两部分：

```cpp
// 1. 核函数实现
__global__ void vec_add(float* x, float* y, float* z, int vector_length)
{
    // 第一步：计算当前线程对应的全局元素索引
    int work_index = blockIdx.x * blockDim.x + threadIdx.x;

    // 第二步：用全局索引读取数据、计算、写回
    if (work_index < vector_length)
    {
        z[work_index] = x[work_index] + y[work_index];
    }
}

// 2. Host 侧调用示例
int main()
{
    // 假设 Host 和 Device 侧的内存分配与初始化均已完成
    int vector_length = 1000;

    // 经验配置：每个线程块 256 个线程
    int threads = 256;

    // 按「向上取整」计算需要的线程块数： (length + threads - 1) / threads
    int blocks = (vector_length + threads - 1) / threads; // 结果为 4

    // 启动核函数：4 个线程块、每个线程块 256 个线程
    vec_add<<<blocks, threads, 0, stream>>>(dev_x, dev_y, dev_z, vector_length);

    // ... 后续的 stream 同步和资源释放 ...
    return 0;
}
```

可以看到，一个典型的一维 SIMT 核函数就是这样的结构：

1. **算全局索引**：用内置变量算出 `work_index`，确定当前线程负责哪个元素。
2. **处理数据**：用 `work_index` 从 Global Memory 读入数据、计算、写回结果。

Host 侧则负责配置并行规模（多少线程块、每块多少线程）并发起调用。掌握这套写法，就能写出一个基础的 SIMT 算子了。

### 一个容易忽略的细节：边界检查

回头看上面核函数里的这行 `if (work_index < vector_length)`，它叫**边界检查**，是实际开发中很容易忽略、却非常重要的一步。

为什么需要它？因为**数据总长度往往无法被总线程数整除**。本例要处理 1000 个元素，按每块 256 线程、向上取整算出 4 个线程块，实际下发了 `4 × 256 = 1024` 个物理线程。当执行到最后一个线程块（`blockIdx.x = 3`）时，末尾 24 个线程算出的 `work_index` 落在 `[1000, 1023]`——已经超出了有效数据范围。

如果没有这道检查，这 24 个线程会去访问 `x[1000]`…`x[1023]` 等不存在的位置，造成**内存越界**。加上 `if (work_index < vector_length)` 后，这些「多余」的线程会直接跳过计算、安全退出，避免了越界风险。

所以一个更完整的核函数写法是「**算全局索引 → 边界检查 → 处理数据**」这三步。在数据长度不确定或不能被整除时，边界检查几乎是必须的。

### 术语速查

| 术语 | 说明 |
| --- | --- |
| 核函数 | 在 AI 处理器上并行执行、由 Host 发起调用的 device 侧入口函数。 |
| `__global__` | 标识函数是核函数的类型限定符。 |
| `__aicore__` | 标识除核函数外的 device 侧执行函数。 |
| `<<<...>>>` | Host 侧调用核函数时使用的执行配置语法。 |

## 小节小结

本小节讲解了如何编写 SIMT 算子的核心——核函数：

- **定义**：用 `__global__` 标识，返回类型必须是 `void`，计算结果通过指针参数写回 Global Memory，而不是用 return。
- **调用关系**：Host 函数通过 `<<<...>>>` 调用核函数；核函数是进入 Device 并行计算的唯一入口，可调用 `__aicore__` 标识的 device 函数，但不能调用另一个核函数。
- **执行配置**：`<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>`，前两个参数决定并行规模（对应 `gridDim`、`blockDim`）。
- **定位数据**：核函数内用内置变量 `blockIdx`、`blockDim`、`threadIdx` 算出全局索引 `work_index`，让每个线程处理自己那一份数据。
- **边界检查**：数据长度常无法被线程块整除，需多分配线程并用 `if (work_index < length)` 拦截越界线程。

掌握「定义 → 调用 → 算全局索引 → 边界检查」这条主线，就具备了读写一个基础 SIMT 算子的能力。而要写出**高性能**的算子，还需要理解数据在不同存储介质间如何流动、如何配置——这正是下一章节《内存层级》要讲的内容。

## 课后练习

本节讲解了 SIMT 核函数的定义、调用、内置变量定位数据和边界检查，请根据学习内容完成以下题目进行自测。

1. （判断题）核函数的返回类型可以是 `int`，用于把计算结果返回给 Host 侧。

2. （单选题）定义核函数时，必须使用哪个函数类型限定符？  
    A. `__aicore__`  
    B. `__global__`  
    C. `__ubuf__`  
    D. `__device__`  

3. （单选题）要处理 1000 个元素、每块 256 个线程，实际下发 1024 个线程，核函数中 `if (work_index < vector_length)` 这行边界检查的作用是？  
    A. 计算线程的全局索引  
    B. 让多余的 24 个越界线程安全退出，防止内存越界  
    C. 提高线程并行度  
    D. 分配动态共享内存  

4. （多选题）以下关于核函数的说法，哪些是正确的？  
    A. Host 侧函数通过 `<<<...>>>` 调用核函数  
    B. 核函数可以调用用 `__aicore__` 标识的 device 侧函数，但不能调用另一个核函数  
    C. 核函数可以在内部直接调用另一个核函数，实现多级并行  
    D. 核函数返回类型必须是 `void`，计算结果通过指针参数写回  

**执行以下代码获取答案。**

## 参考答案

运行下面的单元格查看课后练习参考答案。


In [ ]:
!cat answer/03.04.03_answer.txt
